In [1]:
import os
# 1. Use single GPU (0) to avoid PyTorch DataParallel NCCL broadcast failure on multi-GPU nodes
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# 2. Enable NCCL debug logging for troubleshooting
os.environ["NCCL_DEBUG"] = "INFO"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

import torch
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

print("Device configured:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


/nuvodata/User_data/ak57139k/anmolpro/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device configured: NVIDIA H100 80GB HBM3


/nuvodata/User_data/ak57139k/anmolpro/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
dataset = load_dataset("stanfordnlp/imdb")

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [4]:
train_dataset = dataset['train'].select(range(1000))  # Use a smaller subset for quick training
test_dataset = dataset['test'].select(range(500))  # Use a smaller subset for quick evaluation

In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [6]:
# get the total number of features dataset have
len(dataset['train'].features)  # Display the first training example
dataset['train'].features

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

In [7]:
# Tokenization function
def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

In [8]:
dataset['train'][0]  # Display the first training example

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [9]:
# Apply tokenization + rename + format in a single flow
def preprocess(ds):
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])  # remove raw text (saves memory)
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

In [10]:
train_dataset = preprocess(train_dataset)

In [11]:
test_dataset = preprocess(test_dataset)

In [12]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12151.74it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

In [13]:
for layer in model.bert.encoder.layer:
    print(layer)

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)
BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (qu

In [14]:
import numpy as np
from transformers import Trainer, TrainingArguments

# Compute metrics for evaluation logging
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": float(accuracy)}

training_args = TrainingArguments(
    output_dir="./bert_finetuned_imdb",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=10,         # Print training loss every 10 steps
    eval_strategy="steps",     # Run evaluation every 25 steps
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",         # Terminal/notebook logging without external services
)


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


In [16]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
25,0.047984,0.008922,1.000000
50,0.003372,0.002406,1.000000
75,0.002124,0.001559,1.000000
100,0.001553,0.001274,1.000000
125,0.001452,0.001190,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


TrainOutput(global_step=125, training_loss=0.030180202350020408, metrics={'train_runtime': 13.3364, 'train_samples_per_second': 74.983, 'train_steps_per_second': 9.373, 'total_flos': 131555527680000.0, 'train_loss': 0.030180202350020408, 'epoch': 1.0})

In [17]:
trainer.save_model("./bert-finetuned-imdb")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.19it/s]


In [18]:
tokenizer.save_pretrained("./bert-finetuned-imdb") # Save tokenizer for future inference

('./bert-finetuned-imdb/tokenizer_config.json',
 './bert-finetuned-imdb/tokenizer.json')

In [19]:
metrics = trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
0.001452,0.001190,125,1.000000


Prediction

In [20]:
tokenizer = BertTokenizer.from_pretrained('bert-finetuned-imdb')
model = BertForSequenceClassification.from_pretrained('bert-finetuned-imdb')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11581.54it/s]


In [22]:
from transformers import pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

In [23]:
test = "This movie was fantastic! The acting was superb and the plot was gripping."
result = classifier(test)

In [24]:
result

[{'label': 'LABEL_0', 'score': 0.9975649118423462}]

Publishing to Hugging Face

In [28]:
import os
from dotenv import load_dotenv, find_dotenv
from huggingface_hub import login, HfApi, create_repo

# 1. Load token from .env (Full Access / Write Token)
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(".env")
load_dotenv("SLM_Experiment/.env")

token = os.getenv("HUGGINGFACE_FULL_ACCESS_TOKEN") or os.getenv("HUGGINGFACE_WRITE_TOKEN")
if not token:
    raise ValueError("No write/full access token found in .env!")

login(token=token, add_to_git_credential=True)
api = HfApi()
username = api.whoami()["name"]
print(f"Logged in as: {username}")


Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Logged in as: Akhand108


In [29]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '673ef604fce7a1e1c97bc685', 'name': 'Akhand108', 'fullname': 'Akhand Pratap Singh', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/6z894xaxshMIJuHqi3QWZ.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'FullAccessToken', 'role': 'fineGrained', 'createdAt': '2026-09-09T15:06:46.698Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write', 'post.write'], 'scoped': [{'entity': {'_id': '673ef604fce7a1e1c97bc685', 'type': 'user', 'name': 'Akhand108'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write', 'inference.serverless.write', 'inference.endpoints.infer.write', 'inference.endpoints.write', 'user.webhooks.read', 'user.webhooks.write', 'collection.read', 'collection.write', 'discussion.write', 'user.billing.read', 'job.write', 'user.notifications.read', 'user.notifications.write']}

In [30]:
# 2. Define repository name under your account namespace
repo_name = "my-bert-imdb2"
repo_id = f"{username}/{repo_name}"  # e.g., Akhand108/my-bert-imdb2

print(f"Target Hugging Face Repo: https://huggingface.co/{repo_id}")

# 3. Create repository on Hugging Face (if it does not exist)
create_repo(repo_id=repo_id, exist_ok=True, token=token)

# 4. Push model and tokenizer to your repo
tokenizer.push_to_hub(repo_id, token=token)
model.push_to_hub(repo_id, token=token)

print(f"Successfully published model to: https://huggingface.co/{repo_id}")


Target Hugging Face Repo: https://huggingface.co/Akhand108/my-bert-imdb2


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]
Processing Files (1 / 1): 100%|██████████|  438MB /  438MB, 19.5MB/s  
New Data Upload: 100%|██████████|  436MB /  436MB, 19.4MB/s  


Successfully published model to: https://huggingface.co/Akhand108/my-bert-imdb2
